In [1]:
# %load EO4_grating_ring.py
import os

from phidl import Device, Layer, make_device, Path, CrossSection
from phidl import quickplot as qp
from phidl import set_quickplot_options
import phidl.geometry as pg
import phidl.routing as pr
import phidl.path as pp
from phidl.utilities import write_svg
import gdspy
from librtwu import racetrack_euler_coupling_pulley
from librtwu import euler_bend
from librtwu import wg
from librtwu import arc
from librtwu import alignment_mark
from librtwu import alignment_mark_array
from librtwu import racetrack_electrode
from librtwu import generate_poling_finger_z_cut
from librtwu import s20_50_global_marker
from librtwu import sqm
from librtwu import s20_racetrack_marker
from librtwu import ring_resonator

from librtwu import TE_grating_coupler

import scipy.io
import scipy.special as sc
from scipy.constants import pi
from matplotlib import pyplot as plt
import numpy as np
from scipy.spatial.distance import euclidean
from datetime import *
import operator

# %%
# layer definition
layer_poling_finger = 0
layer_global_marker = 1
layer_poling_lead = 0
layer_protect_gloabe_marker=2
layer_ring_resonator = 23   #  used
layer_racetrack_electrode = 4  #  grating
layer_label = 24            #  used
layer_resonator_marker = 6
layer_ring_marker = 25      #  used
layer_ring_global_marker = 9
w_ring_out = 0.51
w_ring_in = 0.55
w_wg_out = 0.51
w_wg_in = 0.55
R_ring_out = 88
R_ring_in = 87.216
gap_s1 = 0.2
gap_s2 = 1
gap_e = 6
num = 14
detaX = 700     # 不同环之间的x错位
detaY = 0      # 不同环之间的y错位
detaY2 = -1000      # 不同环之间的y错位
Dis_fiber_array=3*127  #光纤之间的间隔127um
L_ytop = 3000
L_ybottom = 3000

# %%

gap = np.linspace(0.2, 1.5, 14)  # gap随着曲线不同位置变化
list_idx = range(num)
D = Device()

#外环扫gap
for idx in list_idx:

    """
    创建一个euler racetrack
    """
    # R_eul coupling区域曲率，R_arc coupling区域圆环半径， angle_eu_c 耦合区域欧拉曲线出射角度，p欧拉曲线使用的x方向长度占总的angle_eu_c-90度范围内x的长度
    Dring = ring_resonator(width=w_ring_out, R_ring=R_ring_out, gap_s=gap[idx], gap_e=gap_e, layer=layer_ring_resonator)
    dring = D << Dring.rotate(-90,center=(0, 0))

    txt = 'g={:.2f}\n r=88'.format(gap[idx])
    yc = -310       # 设定text距离最近的racetrack 2mm多
    xc = 80
    t = D.add_ref(pg.text(text=txt, size=20, justify='right', layer=layer_label))
    t.move((xc, yc))

    """
    上右部走线接出
    """
    l_wg1=20
    WG1 = wg(width1=w_wg_out, width2=w_wg_out, length=l_wg1, layer=layer_ring_resonator)
    wg1 = D << WG1
    wg1.connect(port=2, destination=dring.ports[1])

    r_arc1=1.3*R_ring_out
    ARC = arc(width1=w_wg_out, width2=w_wg_out, angle1=0, angle2=90, R0=r_arc1,layer=layer_ring_resonator)
    a_left1 = D << ARC
    a_left1.connect(port=2, destination=wg1.ports[1])

    l_wg2 = Dis_fiber_array-2*r_arc1
    WG2 = wg(width1=w_wg_out, width2=w_wg_out, length=l_wg2, layer=layer_ring_resonator)
    wg2 = D << WG2
    wg2.connect(port=2, destination=a_left1.ports[1])

    ARC2 = arc(width1=w_wg_out, width2=w_wg_out, angle1=0, angle2=90, R0=r_arc1,layer=layer_ring_resonator)
    a_left2 = D << ARC2.mirror((0, 0), (1, 0))
    a_left2.connect(port=1, destination=wg2.ports[1])

    l_wg3 = a_left2.ymin-dring.ymin
    WG3 = wg(width1=w_wg_out, width2=w_wg_out, length=l_wg3, layer=layer_ring_resonator)
    wg3 = D << WG3
    wg3.connect(port=1, destination=a_left2.ports[2])

    l_wg4 = 150
    WG4 = wg(width1=w_wg_out, width2=0.8, length=l_wg4, layer=layer_ring_resonator)
    wg4 = D << WG4
    wg4.connect(port=1, destination=wg3.ports[2])

    GC1=TE_grating_coupler(gap = 0.4)
    gc1 = D << GC1
    gc1.connect(port=1, destination=wg4.ports[2])

    """
    下部走线接出
    """
    l_wg5 = l_wg4
    WG5 = wg(width1=w_wg_out, width2=0.8, length=l_wg5, layer=layer_ring_resonator)
    wg5 = D << WG5
    wg5.connect(port=1, destination=dring.ports[2])

    GC2=TE_grating_coupler(gap = 0.4)
    gc2 = D << GC2
    gc2.connect(port=1, destination=wg5.ports[2])

    """
    ring local marker
    """
    racetrack_marker1 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=1, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker2 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=2, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker3 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=3, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker4 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=4, x0=0, y0=0, layer=layer_ring_marker)  #
    D << racetrack_marker1.move((50, -90))
    # D1 << racetrack_marker2.move((50, 0))
    # D1 << racetrack_marker3.move((50, 0))
    # D1 << racetrack_marker4.move((50, 0))

    """
    略微错开各个ring
    """
    D.move((detaX, detaY))

D.move((-num*detaX, detaY2))
#内环扫gap
for idx in list_idx:

    """
    创建一个euler racetrack
    """
    # R_eul coupling区域曲率，R_arc coupling区域圆环半径， angle_eu_c 耦合区域欧拉曲线出射角度，p欧拉曲线使用的x方向长度占总的angle_eu_c-90度范围内x的长度
    Dring = ring_resonator(width=w_ring_in, R_ring=R_ring_in, gap_s=gap[idx], gap_e=gap_e, layer=layer_ring_resonator)
    dring = D << Dring.rotate(-90,center=(0, 0))

    txt = 'g={:.2f}\n r=87.216'.format(gap[idx])
    yc = -310       # 设定text距离最近的racetrack 2mm多
    xc = 80
    t = D.add_ref(pg.text(text=txt, size=20, justify='right', layer=layer_label))
    t.move((xc, yc))

    """
    上右部走线接出
    """
    l_wg1=20
    WG1 = wg(width1=w_wg_in, width2=w_wg_in, length=l_wg1, layer=layer_ring_resonator)
    wg1 = D << WG1
    wg1.connect(port=2, destination=dring.ports[1])

    r_arc1=1.3*R_ring_out
    ARC = arc(width1=w_wg_in, width2=w_wg_in, angle1=0, angle2=90, R0=r_arc1,layer=layer_ring_resonator)
    a_left1 = D << ARC
    a_left1.connect(port=2, destination=wg1.ports[1])

    l_wg2 = Dis_fiber_array-2*r_arc1
    WG2 = wg(width1=w_wg_in, width2=w_wg_in, length=l_wg2, layer=layer_ring_resonator)
    wg2 = D << WG2
    wg2.connect(port=2, destination=a_left1.ports[1])

    ARC2 = arc(width1=w_wg_in, width2=w_wg_in, angle1=0, angle2=90, R0=r_arc1,layer=layer_ring_resonator)
    a_left2 = D << ARC2.mirror((0, 0), (1, 0))
    a_left2.connect(port=1, destination=wg2.ports[1])

    l_wg3 = a_left2.ymin-dring.ymin
    WG3 = wg(width1=w_wg_in, width2=w_wg_in, length=l_wg3, layer=layer_ring_resonator)
    wg3 = D << WG3
    wg3.connect(port=1, destination=a_left2.ports[2])

    l_wg4 = 150
    WG4 = wg(width1=w_wg_in, width2=0.8, length=l_wg4, layer=layer_ring_resonator)
    wg4 = D << WG4
    wg4.connect(port=1, destination=wg3.ports[2])

    GC1=TE_grating_coupler(gap = 0.4)
    gc1 = D << GC1
    gc1.connect(port=1, destination=wg4.ports[2])

    """
    下部走线接出
    """
    l_wg5 = l_wg4
    WG5 = wg(width1=w_wg_in, width2=0.8, length=l_wg5, layer=layer_ring_resonator)
    wg5 = D << WG5
    wg5.connect(port=1, destination=dring.ports[2])

    GC2=TE_grating_coupler(gap = 0.4)
    gc2 = D << GC2
    gc2.connect(port=1, destination=wg5.ports[2])

    """
    ring local marker
    """
    racetrack_marker1 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=1, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker2 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=2, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker3 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=3, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker4 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=4, x0=0, y0=0, layer=layer_ring_marker)  #
    D << racetrack_marker1.move((50, -90))
    # D1 << racetrack_marker2.move((50, 0))
    # D1 << racetrack_marker3.move((50, 0))
    # D1 << racetrack_marker4.move((50, 0))

    """
    略微错开各个ring
    """
    D.move((detaX, detaY))

D.move((-num*detaX, detaY2))
#内外环扫gap
for idx in list_idx:

    """
    创建一个euler racetrack
    """
    # R_eul coupling区域曲率，R_arc coupling区域圆环半径， angle_eu_c 耦合区域欧拉曲线出射角度，p欧拉曲线使用的x方向长度占总的angle_eu_c-90度范围内x的长度
    Dring = ring_resonator(width=w_ring_out, R_ring=R_ring_out, gap_s=gap[idx], gap_e=gap_e, layer=layer_ring_resonator)
    dring = D << Dring.rotate(-90,center=(0, 0))

    Ring_in = arc(width1=w_ring_in, width2=w_ring_in, angle1=0, angle2=360, R0=R_ring_in, layer=layer_ring_resonator)
    ring_in = D << Ring_in.move((0,-R_ring_in))


    txt = 'g={:.2f}\n concentric'.format(gap[idx])
    yc = -310       # 设定text距离最近的racetrack 2mm多
    xc = 80
    t = D.add_ref(pg.text(text=txt, size=20, justify='right', layer=layer_label))
    t.move((xc, yc))

    """
    上右部走线接出
    """
    l_wg1=20
    WG1 = wg(width1=w_wg_out, width2=w_wg_out, length=l_wg1, layer=layer_ring_resonator)
    wg1 = D << WG1
    wg1.connect(port=2, destination=dring.ports[1])

    r_arc1=1.3*R_ring_out
    ARC = arc(width1=w_wg_out, width2=w_wg_out, angle1=0, angle2=90, R0=r_arc1,layer=layer_ring_resonator)
    a_left1 = D << ARC
    a_left1.connect(port=2, destination=wg1.ports[1])

    l_wg2 = Dis_fiber_array-2*r_arc1
    WG2 = wg(width1=w_wg_out, width2=w_wg_out, length=l_wg2, layer=layer_ring_resonator)
    wg2 = D << WG2
    wg2.connect(port=2, destination=a_left1.ports[1])

    ARC2 = arc(width1=w_wg_out, width2=w_wg_out, angle1=0, angle2=90, R0=r_arc1,layer=layer_ring_resonator)
    a_left2 = D << ARC2.mirror((0, 0), (1, 0))
    a_left2.connect(port=1, destination=wg2.ports[1])

    l_wg3 = a_left2.ymin-dring.ymin
    WG3 = wg(width1=w_wg_out, width2=w_wg_out, length=l_wg3, layer=layer_ring_resonator)
    wg3 = D << WG3
    wg3.connect(port=1, destination=a_left2.ports[2])

    l_wg4 = 150
    WG4 = wg(width1=w_wg_out, width2=0.8, length=l_wg4, layer=layer_ring_resonator)
    wg4 = D << WG4
    wg4.connect(port=1, destination=wg3.ports[2])

    GC1=TE_grating_coupler(gap = 0.4)
    gc1 = D << GC1
    gc1.connect(port=1, destination=wg4.ports[2])

    """
    下部走线接出
    """
    l_wg5 = l_wg4
    WG5 = wg(width1=w_wg_out, width2=0.8, length=l_wg5, layer=layer_ring_resonator)
    wg5 = D << WG5
    wg5.connect(port=1, destination=dring.ports[2])

    GC2=TE_grating_coupler(gap = 0.4)
    gc2 = D << GC2
    gc2.connect(port=1, destination=wg5.ports[2])

    """
    ring local marker
    """
    racetrack_marker1 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=1, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker2 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=2, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker3 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=3, x0=0, y0=0, layer=layer_ring_marker)  #
    # racetrack_marker4 = s20_racetrack_marker(size1=20,distance_x=150,distance_y=400, gap=40, gap2=80, region=4, x0=0, y0=0, layer=layer_ring_marker)  #
    D << racetrack_marker1.move((50, -90))
    # D1 << racetrack_marker2.move((50, 0))
    # D1 << racetrack_marker3.move((50, 0))
    # D1 << racetrack_marker4.move((50, 0))

    """
    略微错开各个ring
    """
    D.move((detaX, detaY))

set_quickplot_options(show_ports=True,show_subports=True,label_aliases=True,new_window=True,blocking=True,zoom_factor=True,interactive_zoom=True)
qp(D)
D.write_gds('E:\Project_Code\E04_grating_ring_20240829.gds', unit=1e-6, precision=1e-9)

# print("BUS_GAP1[1] 的值为:", BUS_GAP1_value)
# print("racetrack straight waveguide length:", l_wg)
# print("racetrack euler waveguide length:", euler_len_total)


FileNotFoundError: [Errno 2] No such file or directory: 'Grating_model_wu.gds'